# 10 — Entity extraction inputs

Prepare a deterministic development sample from the **actual local Djinni CV/JD data**. This notebook validates the frozen A2 partition before selecting 1,000 training CVs and 1,000 training JDs (seed 0). It never samples evaluation documents and does not use `Primary Keyword` or fit labels as extraction gold.

Approved delivery: [entity extraction notebooks plan](../plan/2026-10-05-entity-extraction-notebooks/README.md), based on [the original design](../plan/2026-09-13-entity-extraction/README.md). Run with the installed package using `uv run jupyter nbconvert --to notebook --execute --inplace notebooks/10-entity-extraction-inputs.ipynb`. Each notebook starts from its own imports and rebuilds required artifacts through package calls.

In [1]:
import json
import pandas as pd
from candidate_screener.config import PROJECT_ROOT

# Aggregate outputs only: cap every saved output at 2,000 characters.
def show_summary(value):
    rendered = json.dumps(value, indent=2, default=str)
    print(rendered if len(rendered) <= 1900 else rendered[:1850] + "\n[output truncated]")

SAMPLE_SIZE = 1000  # per document type, exclusively from the frozen A2 train region
SEED = 0


## Validate local sources and select documents

The package checks raw ID uniqueness, exact partition coverage and train membership. Each source field remains separate so an evidence span cannot cross a fabricated field boundary. Missing mandatory A2 inputs fail immediately. Optional sources are inventoried by local file presence; historical source-card counts do not establish local availability.

In [2]:
from candidate_screener.extraction.inputs import prepare_inputs

prepared = prepare_inputs(sample_size=SAMPLE_SIZE, seed=SEED)
show_summary(prepared["summary"])


{
  "cv": {
    "raw_documents": 210250,
    "eligible_documents": 157688,
    "sampled_documents": 1000,
    "fields": [
      "Position",
      "Moreinfo",
      "Looking For",
      "Highlights"
    ]
  },
  "jd": {
    "raw_documents": 141897,
    "eligible_documents": 106423,
    "sampled_documents": 1000,
    "fields": [
      "Position",
      "Long Description"
    ]
  }
}


In [3]:
show_summary({key: value for key, value in prepared["availability"].items() if key in {"djinni-cv", "djinni-jd"}})

{
  "djinni-cv": {
    "catalog_id": "A2",
    "status": "present_unvalidated",
    "missing_artifacts": []
  },
  "djinni-jd": {
    "catalog_id": "A2",
    "status": "present_unvalidated",
    "missing_artifacts": []
  }
}


In [4]:
show_summary({key: value for key, value in prepared["availability"].items() if key not in {"djinni-cv", "djinni-jd"}})

{
  "dataturks": {
    "catalog_id": "B1",
    "status": "missing",
    "missing_artifacts": [
      "traindata.json",
      "testdata.json"
    ]
  },
  "skillspan": {
    "catalog_id": "B2",
    "status": "missing",
    "missing_artifacts": [
      "*-train-*.parquet",
      "*-validation-*.parquet",
      "*-test-*.parquet"
    ]
  },
  "esco": {
    "catalog_id": "D1",
    "status": "missing",
    "missing_artifacts": [
      "skills_en.csv",
      "occupations_en.csv",
      "occupationSkillRelations_en.csv"
    ]
  },
  "data-jobs": {
    "catalog_id": "D2",
    "status": "missing",
    "missing_artifacts": [
      "*-train-*.parquet"
    ]
  }
}


## Provenance and privacy

Preparation records source and partition hashes, deterministic selected document IDs and the chosen fields. Original text stays local in memory; it is never displayed here. The extraction run in notebook 11 writes its sample manifest and provenance under ignored `data/processed/extraction/`.

CV `Position` is a profile title, not employment history. `Looking For` records aspirations, not applied experience. Null fields are handled by the package. Neither corpus supplies human-labelled extraction spans, so these inputs support descriptive coverage and operational checks only.

**Open gates:** B1 DataTurks/B2 SkillSpan restoration and validated adapters, ESCO vocabulary integration, relation/date linking and supervised modelling. No downloads or new annotation occur in this delivery.